# SWS3022 Lecture 1 Lab: From Stock Data to Trading Signals

This notebook supports Lecture 1 of **SWS3022 AI/ML for Financial Services**.

You will analyze one stock using daily price data. The goal is not to build a profitable trading strategy. The goal is to practice a repeatable workflow:

```text
question -> data -> cleaning -> features -> visualization -> model/rule -> evaluation -> dashboard/demo
```

A technical indicator is not a complete trading strategy. Bollinger Band signals are used here for learning purposes only.

## 1. Setup

Install `yfinance` if needed, then import the libraries used in this lab.

In [ ]:
!pip install yfinance

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (12, 5)

## 2. Choose a Ticker

Change the ticker if you want to analyze another stock. Examples: `MSFT`, `TSLA`, `NVDA`, `JPM`.

In [ ]:
ticker = "AAPL"

## 3. Download Daily Data

This notebook uses Yahoo Finance data for learning. Document the ticker, date range, download date, and data limitations.

For simplicity, this notebook uses `Close`. For more careful financial analysis, you may consider `Adj Close`, which adjusts for corporate actions such as splits and dividends.

Recent `yfinance` versions may return multi-level columns even for one ticker. The next code cell flattens the downloaded data back to ordinary columns such as `Close`, `Adj Close`, `High`, `Low`, `Open`, and `Volume` when needed. If your columns still look different, inspect `df.columns` and adjust the column selection accordingly.

In [ ]:
start_date = "2024-01-01"
end_date = "2025-01-01"

df = yf.download(ticker, start=start_date, end=end_date, auto_adjust=False)

# Recent yfinance versions may return multi-level columns even for one ticker.
# Convert them back to ordinary columns such as Close, Adj Close, High, Low, Open, Volume.
if isinstance(df.columns, pd.MultiIndex):
    ticker_upper = ticker.upper()
    last_level = list(df.columns.get_level_values(-1))
    last_level_upper = [str(value).upper() for value in last_level]

    if ticker_upper in last_level_upper:
        selected_ticker = last_level[last_level_upper.index(ticker_upper)]
        df = df.xs(selected_ticker, axis=1, level=-1)
    else:
        first_level = list(df.columns.get_level_values(0))
        first_level_upper = [str(value).upper() for value in first_level]
        if ticker_upper in first_level_upper:
            selected_ticker = first_level[first_level_upper.index(ticker_upper)]
            df = df.xs(selected_ticker, axis=1, level=0)

# Make sure downstream assignments do not modify a view.
df = df.copy()
df.head()

## 4. Inspect the Dataset

Before calculating indicators, check the shape, columns, missing values, and date range.

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.index.min(), df.index.max()

**Reflection**

- What columns are available?
- What is the date range?
- Are there missing values?
- Is this dataset sufficient for a simple first analysis?

Write your answer here:

> 

## 5. Plot Closing Price

The closing price plot shows the price level over time.

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(df.index, df["Close"], label="Close")
plt.title(f"{ticker} Closing Price in 2024")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.show()

**Reflection**

- Did the price mostly increase, decrease, or move sideways during the year?
- Can you identify periods of sharp movement?

Write your answer here:

> 

## 6. Compute Daily Returns

Raw prices show levels. Returns show relative changes. Returns are often more useful for comparison and risk analysis.

In [ ]:
df["Return"] = df["Close"].pct_change()

plt.figure(figsize=(12, 4))
plt.plot(df.index, df["Return"], label="Daily return")
plt.axhline(0, color="black", linewidth=1)
plt.title(f"{ticker} Daily Returns in 2024")
plt.xlabel("Date")
plt.ylabel("Daily Return")
plt.legend()
plt.show()

**Reflection**

- Why might returns be more useful than raw prices when comparing different stocks?
- Which period looks most volatile?

Write your answer here:

> 

## 7. Compute Log Returns

Log returns are commonly used in quantitative finance. For this introductory lab, it is enough to know that both simple returns and log returns describe price changes rather than price levels.

In [ ]:
df["LogReturn"] = np.log(df["Close"] / df["Close"].shift(1))
df[["Close", "Return", "LogReturn"]].head()

## 8. Add a 20-Day Moving Average

A moving average smooths short-term fluctuations in the price series.

In [ ]:
df["MA20"] = df["Close"].rolling(20).mean()

plt.figure(figsize=(12, 5))
plt.plot(df.index, df["Close"], label="Close")
plt.plot(df.index, df["MA20"], label="20-day moving average")
plt.title(f"{ticker} Close and 20-Day Moving Average")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.show()

**Reflection**

- How does the moving average differ from the original closing price?
- What information is lost when we smooth the price series?

Write your answer here:

> 

## 9. Compute Rolling Volatility

Rolling volatility is calculated on returns, not prices. It measures how unstable recent returns have been.

In [ ]:
df["Volatility20"] = df["Return"].rolling(20).std()

plt.figure(figsize=(12, 4))
plt.plot(df.index, df["Volatility20"], label="20-day rolling volatility")
plt.title(f"{ticker} 20-Day Rolling Volatility")
plt.xlabel("Date")
plt.ylabel("Volatility")
plt.legend()
plt.show()

**Reflection**

What does high rolling volatility mean for an investor or risk manager?

Write your answer here:

> 

## 10. Compute Drawdown

Drawdown measures decline from a previous cumulative peak. It helps us understand the painful path an investor might experience, not only the final return.

In [ ]:
cum_return = (1 + df["Return"]).cumprod()
running_max = cum_return.cummax()
df["Drawdown"] = cum_return / running_max - 1

plt.figure(figsize=(12, 4))
plt.plot(df.index, df["Drawdown"], label="Drawdown")
plt.title(f"{ticker} Drawdown in 2024")
plt.xlabel("Date")
plt.ylabel("Drawdown")
plt.legend()
plt.show()

**Reflection**

Why is drawdown important even when final return is positive?

Write your answer here:

> 

## 11. Build Bollinger Bands

Bollinger Bands are drawn around a moving average of price.

```text
Middle Band = 20-day moving average
Upper Band = Middle Band + 2 x rolling standard deviation of price
Lower Band = Middle Band - 2 x rolling standard deviation of price
```

This is different from rolling volatility, which is usually calculated on returns.

In [ ]:
window = 20

df["Middle"] = df["Close"].rolling(window).mean()
df["Std20"] = df["Close"].rolling(window).std()
df["Upper"] = df["Middle"] + 2 * df["Std20"]
df["Lower"] = df["Middle"] - 2 * df["Std20"]

df[["Close", "Middle", "Upper", "Lower"]].head(25)

The first rows contain missing values because a 20-day rolling window needs 20 observations.

`df_clean` is useful for later modeling or backtesting. In this lab, we may still use `df` for visualization so that the full date range remains visible.

In [ ]:
df_clean = df.dropna().copy()
df_clean.head()

## 12. Visualize Bollinger Bands

The bands become wider when recent price variation is larger and narrower when recent price variation is smaller.

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(df.index, df["Close"], label="Close")
plt.plot(df.index, df["Middle"], label="20-day MA")
plt.plot(df.index, df["Upper"], linestyle="--", label="Upper Band")
plt.plot(df.index, df["Lower"], linestyle="--", label="Lower Band")
plt.title(f"{ticker} Bollinger Bands in 2024")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.show()

**Reflection**

- When do the bands become wider or narrower?
- What does a wider band suggest about recent price variation?

Write your answer here:

> 

## 13. Generate Simple Indicator Signals

Here, `1` means the price is below the lower band, and `-1` means the price is above the upper band. These are indicator signals, not proven trading instructions.

In [ ]:
df["Signal"] = 0
df.loc[df["Close"] < df["Lower"], "Signal"] = 1
df.loc[df["Close"] > df["Upper"], "Signal"] = -1

buy = df[df["Signal"] == 1]
sell = df[df["Signal"] == -1]

num_buy = (df["Signal"] == 1).sum()
num_sell = (df["Signal"] == -1).sum()

num_buy, num_sell

## 14. Visualize Signals

Use neutral labels. A lower-band signal or upper-band signal is not automatically a trading action.

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(df.index, df["Close"], label="Close")
plt.plot(df.index, df["Middle"], label="20-day MA")
plt.plot(df.index, df["Upper"], linestyle="--", label="Upper Band")
plt.plot(df.index, df["Lower"], linestyle="--", label="Lower Band")

plt.scatter(buy.index, buy["Close"], marker="^", label="Lower-band signal")
plt.scatter(sell.index, sell["Close"], marker="v", label="Upper-band signal")

plt.title(f"{ticker} Bollinger Band Signals in 2024")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.show()

## 15. Indicator vs Strategy

A technical indicator is not the same as a complete trading strategy.

A strategy would also need:

- Entry rules
- Exit rules
- Position sizing
- Transaction cost assumptions
- Risk management
- Evaluation on unseen data

**Warning:** Do not claim that Bollinger Bands guarantee profit. This notebook only demonstrates feature engineering, visualization, and signal design.

## 16. Train/Test Split by Date

For forecasting or trading evaluation, train on earlier data and test on later data. Randomly shuffling financial time series often creates unrealistic evaluation.

In [ ]:
train = df[df.index < "2024-10-01"]
test = df[df.index >= "2024-10-01"]

train.shape, test.shape

## 17. Final Reflection Questions

Please answer the following questions in your notebook:

1. What ticker did you choose?
2. What period did you analyze?
3. How many lower-band and upper-band signals were generated?
4. Do the signals look meaningful on the chart?
5. What are the limitations of this simple approach?
6. What would you improve before using this in a real project?
7. Why is a technical indicator not the same as a trading strategy?
8. Why should time series data usually be split by date instead of randomly?

Write your answers here:

1. 
2. 
3. 
4. 
5. 
6. 
7. 
8. 

## 18. Optional Challenges

1. Change the ticker and compare results.
2. Change the rolling window from 20 to 10 or 50.
3. Compare `Close` and `Adj Close`.
4. Add transaction cost assumptions.
5. Evaluate a simple rule on a later time period.
6. Build a small Streamlit dashboard using the same data.
7. Add another indicator such as RSI or MACD.